#### Step 1: Setup and Data Preparation

In [3]:
%pip install pypdf

import os
import re
import sys
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# Install PDF reader into the notebook kernel if needed
try:
    from pypdf import PdfReader
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pypdf"])
    from pypdf import PdfReader

# ----------------------------
# Config
# ----------------------------
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
assert api_key, "Missing OPENAI_API_KEY in .env"

client = OpenAI(api_key=api_key)

resources = Path("resources")
audio_path = resources / "podcast_small.m4a"
if not audio_path.exists():
    audio_path = resources / "podcast.m4a"

pdf_path = resources / "trustworthy_ai.pdf"

assert audio_path.exists(), f"Missing audio file: {audio_path}"
assert pdf_path.exists(), f"Missing PDF file: {pdf_path}"

def chunk_text(text, chunk_size=1000, overlap=150):
    text = re.sub(r"\s+", " ", text).strip()
    chunks = []
    start = 0
    step = chunk_size - overlap

    while start < len(text):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        start += step

    return chunks

# ----------------------------
# 1) Transcribe podcast audio
# ----------------------------
with audio_path.open("rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-mini-transcribe",
        file=audio_file,
        response_format="text",
    )

transcript_text = getattr(transcript, "text", transcript)
transcript_text = str(transcript_text).strip()

podcast_docs = [
    {
        "text": chunk,
        "metadata": {
            "source_type": "podcast_transcript",
            "source": str(audio_path),
            "filename": audio_path.name,
            "chunk_id": f"podcast_{i}",
            "page": None,
        },
    }
    for i, chunk in enumerate(chunk_text(transcript_text, chunk_size=1000, overlap=150))
]

# ----------------------------
# 2) Load PDF page by page
# ----------------------------
reader = PdfReader(str(pdf_path))
pdf_docs = []

for page_num, page in enumerate(reader.pages, start=1):
    page_text = (page.extract_text() or "").strip()
    if not page_text:
        continue

    for i, chunk in enumerate(chunk_text(page_text, chunk_size=1000, overlap=150)):
        pdf_docs.append(
            {
                "text": chunk,
                "metadata": {
                    "source_type": "eu_ai_act",
                    "source": str(pdf_path),
                    "filename": pdf_path.name,
                    "page": page_num,
                    "chunk_id": f"pdf_p{page_num}_{i}",
                },
            }
        )

# ----------------------------
# 3) Combine and inspect
# ----------------------------
chunks = podcast_docs + pdf_docs

print(f"Transcript characters: {len(transcript_text):,}")
print(f"Podcast chunks: {len(podcast_docs)}")
print(f"PDF chunks: {len(pdf_docs)}")
print(f"Total chunks: {len(chunks)}")
print("\nSample metadata:")
print(chunks[0]["metadata"])
print("\nSample text preview:")
print(chunks[0]["text"][:800])

Note: you may need to restart the kernel to use updated packages.
Transcript characters: 9,915
Podcast chunks: 12
PDF chunks: 197
Total chunks: 209

Sample metadata:
{'source_type': 'podcast_transcript', 'source': 'resources/podcast_small.m4a', 'filename': 'podcast_small.m4a', 'chunk_id': 'podcast_0', 'page': None}

Sample text preview:
So, imagine for a second you're driving across, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out, and demand to see the blueprints, right? You don't interview the welding crew. No, you just... you trust it. You just drive, you trust the bridge, you trust the engineering standards, the inspections, the laws that say this thing won't fail. Right. It's trust in the infrastructure. It's invisible, but it's there. Exactly. But now, let's switch gears. Think about the algorithm that just denied your mortgage application or the AI system scanning your face at the airport. Do you have that same trust? And I mean, hone

#### Step 2: Generate Embeddings and Initial Retrieval

In [4]:
import os
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

# ----------------------------
# Config
# ----------------------------
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
assert api_key, "Missing OPENAI_API_KEY"

client = OpenAI(api_key=api_key)
embedding_model = "text-embedding-3-small"

# ----------------------------
# Prepare texts
# ----------------------------
texts = [item["text"] for item in chunks]
metadatas = [item["metadata"] for item in chunks]

# ----------------------------
# Generate embeddings in batches
# ----------------------------
def embed_texts(text_list, batch_size=64):
    all_embeddings = []

    for start in range(0, len(text_list), batch_size):
        batch = text_list[start:start + batch_size]
        response = client.embeddings.create(
            model=embedding_model,
            input=batch,
        )
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)

    return np.array(all_embeddings, dtype=np.float32)

embeddings = embed_texts(texts, batch_size=64)

# ----------------------------
# Normalize for cosine similarity
# ----------------------------
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings = embeddings / np.where(norms == 0, 1, norms)

print(f"Embedded {len(texts)} chunks")
print(f"Embedding matrix shape: {embeddings.shape}")

# ----------------------------
# Baseline retrieval
# ----------------------------
def retrieve(query, top_k=5):
    query_embedding = client.embeddings.create(
        model=embedding_model,
        input=query,
    ).data[0].embedding

    query_embedding = np.array(query_embedding, dtype=np.float32)
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    scores = embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append(
            {
                "score": float(scores[idx]),
                "text": texts[idx],
                "metadata": metadatas[idx],
            }
        )

    return results

# ----------------------------
# Quick test
# ----------------------------
test_query = "What does the EU AI Act say about high-risk AI systems?"
results = retrieve(test_query, top_k=5)

for i, r in enumerate(results, start=1):
    print(f"\nRank {i} | Score: {r['score']:.4f}")
    print(r["metadata"])
    print(r["text"][:500])

Embedded 209 chunks
Embedding matrix shape: (209, 1536)

Rank 1 | Score: 0.6206
{'source_type': 'eu_ai_act', 'source': 'resources/trustworthy_ai.pdf', 'filename': 'trustworthy_ai.pdf', 'page': 8, 'chunk_id': 'pdf_p8_2'}
es at European, national and international level already apply or are relevant to the development, deployment and use of AI systems today. Legal sources include, but are not limit ed to : EU p rimary law (the Treaties of the European Union and its Charter of Fundamental Rights ), EU s econdary law (such as the General Data Protection Regulation, the Product Liability Directive, the Regulation on the Free Flow of Non -Personal Data, anti-discrimination Directives, consumer law and Safety and Heal

Rank 2 | Score: 0.6157
{'source_type': 'eu_ai_act', 'source': 'resources/trustworthy_ai.pdf', 'filename': 'trustworthy_ai.pdf', 'page': 37, 'chunk_id': 'pdf_p37_4'}
ership in innovative, cut ting-edge AI systems. This ambitious vision will help securing human flourishing of Eur

#### Step 3: Implement Relevance Scoring via an LLM (Optional - Advanced)

In [6]:
import os
import json
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

# ----------------------------
# Config
# ----------------------------
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
assert api_key, "Missing OPENAI_API_KEY"

client = OpenAI(api_key=api_key)

llm_model = "gpt-4o-mini"

# Weighting between semantic similarity and LLM relevance
SIMILARITY_WEIGHT = 0.5
RELEVANCE_WEIGHT = 0.5

# ----------------------------
# Structured scoring function
# ----------------------------
import json

def score_chunk_relevance(query, chunk_text):
    response = client.responses.create(
        model=llm_model,
        input=[
            {
                "role": "system",
                "content": "You are scoring how relevant a document chunk is to a user query. Return only the requested JSON.",
            },
            {
                "role": "user",
                "content": f"""
Query:
{query}

Chunk:
{chunk_text}

Score how relevant this chunk is to the query on a scale from 0 to 100:
- 0 = completely irrelevant
- 100 = directly answers the query

Return a short explanation as well.
""",
            },
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "relevance_score",
                "schema": {
                    "type": "object",
                    "properties": {
                        "score": {
                            "type": "number",
                            "minimum": 0,
                            "maximum": 100
                        },
                        "reason": {
                            "type": "string"
                        }
                    },
                    "required": ["score", "reason"],
                    "additionalProperties": False
                },
                "strict": True
            }
        }
    )

    data = json.loads(response.output_text)
    return float(data["score"]), data["reason"]

# ----------------------------
# Rerank function
# Uses your existing retrieve() from step 2
# ----------------------------
def retrieve_with_llm_rerank(query, top_k=5, candidate_k=10):
    """
    1) Get candidate_k chunks by vector similarity
    2) Ask the LLM to score each candidate
    3) Combine similarity + relevance into one score
    4) Rerank by combined score
    """
    candidates = retrieve(query, top_k=candidate_k)

    reranked = []
    for item in candidates:
        relevance_score, reason = score_chunk_relevance(query, item["text"])

        similarity_score = item["score"]  # cosine similarity from step 2, usually in [-1, 1]
        relevance_norm = relevance_score / 100.0

        combined_score = (
            SIMILARITY_WEIGHT * similarity_score +
            RELEVANCE_WEIGHT * relevance_norm
        )

        reranked.append(
            {
                "combined_score": float(combined_score),
                "similarity_score": float(similarity_score),
                "relevance_score": float(relevance_score),
                "reason": reason,
                "text": item["text"],
                "metadata": item["metadata"],
            }
        )

    reranked = sorted(reranked, key=lambda x: x["combined_score"], reverse=True)
    return reranked[:top_k]

# ----------------------------
# Test it
# ----------------------------
query = "What does the EU AI Act say about high-risk AI systems?"
results = retrieve_with_llm_rerank(query, top_k=5, candidate_k=8)

for i, r in enumerate(results, start=1):
    print(f"\nRank {i}")
    print(f"Combined:   {r['combined_score']:.4f}")
    print(f"Similarity:  {r['similarity_score']:.4f}")
    print(f"Relevance:   {r['relevance_score']:.1f}")
    print(f"Metadata:    {r['metadata']}")
    print(f"Reason:      {r['reason']}")
    print(f"Preview:     {r['text'][:350]}")


Rank 1
Combined:   0.6811
Similarity:  0.6122
Relevance:   75.0
Metadata:    {'source_type': 'eu_ai_act', 'source': 'resources/trustworthy_ai.pdf', 'filename': 'trustworthy_ai.pdf', 'page': 19, 'chunk_id': 'pdf_p19_0'}
Reason:      The chunk discusses aspects related to risk assessment and safety measures for AI systems, which aligns with the concerns addressed in the EU AI Act regarding high-risk AI systems. However, it does not explicitly mention the EU AI Act or its specific regulations on high-risk AI systems.
Preview:     17 This can mean that AI systems switch from a statistical to rule -based procedure, or that they ask for a human operator before continuing their action .39 It must be ensured that the system will do what it is supposed to do without harming living beings or the environment. This includes the minimi sation of unintended consequences and errors. In

Rank 2
Combined:   0.5317
Similarity:  0.6134
Relevance:   45.0
Metadata:    {'source_type': 'eu_ai_act', 'source'

#### Step 4: Implement Reranker (Cohere or Cross-Encoder) (Optional - Advanced)

In [7]:
%pip install sentence-transformers torch

from sentence_transformers import CrossEncoder

# ----------------------------
# Config
# ----------------------------
# A standard reranking model from Sentence-Transformers docs
reranker_model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"

reranker = CrossEncoder(reranker_model_name)

def retrieve_with_cross_encoder(query, top_k=5, candidate_k=10):
    """
    1) Get candidate_k chunks from your embedding retriever
    2) Score query-passage pairs with a cross-encoder
    3) Rerank by cross-encoder score
    """
    candidates = retrieve(query, top_k=candidate_k)

    pairs = [(query, item["text"]) for item in candidates]
    scores = reranker.predict(pairs)

    reranked = []
    for item, score in zip(candidates, scores):
        reranked.append(
            {
                "reranker_score": float(score),
                "similarity_score": float(item["score"]),
                "text": item["text"],
                "metadata": item["metadata"],
            }
        )

    reranked = sorted(reranked, key=lambda x: x["reranker_score"], reverse=True)
    return reranked[:top_k]

# ----------------------------
# Test it
# ----------------------------
query = "What does the EU AI Act say about high-risk AI systems?"
results = retrieve_with_cross_encoder(query, top_k=5, candidate_k=8)

for i, r in enumerate(results, start=1):
    print(f"\nRank {i}")
    print(f"Reranker score:  {r['reranker_score']:.4f}")
    print(f"Similarity score: {r['similarity_score']:.4f}")
    print(f"Metadata:         {r['metadata']}")
    print(f"Preview:          {r['text'][:350]}")

Note: you may need to restart the kernel to use updated packages.


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


Rank 1
Reranker score:  0.3355
Similarity score: 0.6122
Metadata:         {'source_type': 'eu_ai_act', 'source': 'resources/trustworthy_ai.pdf', 'filename': 'trustworthy_ai.pdf', 'page': 19, 'chunk_id': 'pdf_p19_0'}
Preview:          17 This can mean that AI systems switch from a statistical to rule -based procedure, or that they ask for a human operator before continuing their action .39 It must be ensured that the system will do what it is supposed to do without harming living beings or the environment. This includes the minimi sation of unintended consequences and errors. In

Rank 2
Reranker score:  0.1451
Similarity score: 0.6134
Metadata:         {'source_type': 'eu_ai_act', 'source': 'resources/trustworthy_ai.pdf', 'filename': 'trustworthy_ai.pdf', 'page': 4, 'chunk_id': 'pdf_p4_2'}
Preview:          AI systems in a way that adhere s to the ethical principles of : respect for h uman autonomy, prevention of h arm, fairness and explicability. Acknowledge and address the potential 

##### Relevance and Reranker scoring comparative:

The embedding baseline retrieved broadly relevant EU AI Act chunks, but step 3 and step 4 refined the ordering differently. Step 3 produced a hybrid ranking where semantically similar chunks with moderate LLM relevance still stayed high. Step 4, using a cross-encoder reranker, was stricter and created a clearer score separation, pushing weaker borderline chunks down. Both methods agreed on the most relevant chunk, but the cross-encoder gave more precise ranking among the less relevant candidates.

#### Step 5: Add Metadata Filtering

In [8]:
import numpy as np

def matches_filters(metadata, filters=None):
    """
    filters examples:
      {"source_type": "eu_ai_act"}
      {"source_type": "podcast_transcript", "filename": "podcast_small.m4a"}
      {"page": 19}
      {"source_type": ["eu_ai_act", "podcast_transcript"]}
    """
    if not filters:
        return True

    for key, expected in filters.items():
        value = metadata.get(key)

        if isinstance(expected, (list, tuple, set)):
            if value not in expected:
                return False
        else:
            if value != expected:
                return False

    return True

def retrieve_filtered(query, top_k=5, filters=None):
    """
    Retrieve only from chunks that match the metadata filters.
    Uses your existing:
      - embeddings
      - texts
      - metadatas
      - client
      - embedding_model
    """
    query_embedding = client.embeddings.create(
        model=embedding_model,
        input=query,
    ).data[0].embedding

    query_embedding = np.array(query_embedding, dtype=np.float32)
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    candidate_indices = [
        i for i, md in enumerate(metadatas)
        if matches_filters(md, filters)
    ]

    if not candidate_indices:
        return []

    candidate_embeddings = embeddings[candidate_indices]
    scores = candidate_embeddings @ query_embedding
    top_local_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for local_idx in top_local_indices:
        global_idx = candidate_indices[local_idx]
        results.append(
            {
                "score": float(scores[local_idx]),
                "text": texts[global_idx],
                "metadata": metadatas[global_idx],
            }
        )

    return results

# ----------------------------
# Examples
# ----------------------------

query = "What does the EU AI Act say about high-risk AI systems?"

print("EU AI Act only:")
results = retrieve_filtered(query, top_k=5, filters={"source_type": "eu_ai_act"})
for i, r in enumerate(results, start=1):
    print(f"\nRank {i} | Score: {r['score']:.4f}")
    print(r["metadata"])
    print(r["text"][:350])

print("\nPodcast only:")
results = retrieve_filtered(query, top_k=5, filters={"source_type": "podcast_transcript"})
for i, r in enumerate(results, start=1):
    print(f"\nRank {i} | Score: {r['score']:.4f}")
    print(r["metadata"])
    print(r["text"][:350])


EU AI Act only:

Rank 1 | Score: 0.6206
{'source_type': 'eu_ai_act', 'source': 'resources/trustworthy_ai.pdf', 'filename': 'trustworthy_ai.pdf', 'page': 8, 'chunk_id': 'pdf_p8_2'}
es at European, national and international level already apply or are relevant to the development, deployment and use of AI systems today. Legal sources include, but are not limit ed to : EU p rimary law (the Treaties of the European Union and its Charter of Fundamental Rights ), EU s econdary law (such as the General Data Protection Regulation, th

Rank 2 | Score: 0.6157
{'source_type': 'eu_ai_act', 'source': 'resources/trustworthy_ai.pdf', 'filename': 'trustworthy_ai.pdf', 'page': 37, 'chunk_id': 'pdf_p37_4'}
ership in innovative, cut ting-edge AI systems. This ambitious vision will help securing human flourishing of European citizens, both individually and collectively . Our goal is to create a culture of “Trustworthy AI for Europe”, whereby the benefits of AI can be reaped by all in a manner that ensures 

#### Step 6: Complete RAG Pipeline with Reranking

In [11]:
def retrieve_with_cross_encoder_filtered(query, top_k=5, candidate_k=10, filters=None):
    # 1) Get candidate chunks with metadata filtering
    candidates = retrieve_filtered(query, top_k=candidate_k, filters=filters)

    if not candidates:
        return []

    # 2) Score query-passage pairs with the cross-encoder
    pairs = [(query, item["text"]) for item in candidates]
    scores = reranker.predict(pairs)

    # 3) Attach scores and rerank
    reranked = []
    for item, score in zip(candidates, scores):
        reranked.append(
            {
                "reranker_score": float(score),
                "similarity_score": float(item["score"]),
                "text": item["text"],
                "metadata": item["metadata"],
            }
        )

    reranked = sorted(reranked, key=lambda x: x["reranker_score"], reverse=True)
    return reranked[:top_k]


def rag_pipeline(query, top_k=5, candidate_k=10, filters=None):
    # 1) Retrieve + rerank
    results = retrieve_with_cross_encoder_filtered(
        query=query,
        top_k=top_k,
        candidate_k=candidate_k,
        filters=filters,
    )

    # 2) Build context
    context = "\n\n".join(
        f"[Source: {r['metadata'].get('filename', 'unknown')}, Page: {r['metadata'].get('page', 'n/a')}]\n{r['text']}"
        for r in results
    )

    # 3) Ask the model
    prompt = f"""
You are a legal AI assistant. Use only the provided context to answer the question.
If the context is insufficient, say so clearly.

Context:
{context}

Question: {query}
"""

    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt,
    )

    return response.output_text, results


# Example
query = "What does the EU AI Act say about high-risk AI systems?"
answer, sources = rag_pipeline(
    query,
    top_k=5,
    candidate_k=10,
    filters={"source_type": "eu_ai_act"},
)

print(answer)

for i, r in enumerate(sources, start=1):
    print(f"\nSource {i}:")
    print(r["metadata"])
    print(r["text"][:250])

The provided context does not specify details about the EU AI Act and its provisions regarding high-risk AI systems. Therefore, I cannot answer the question clearly.

Source 1:
{'source_type': 'eu_ai_act', 'source': 'resources/trustworthy_ai.pdf', 'filename': 'trustworthy_ai.pdf', 'page': 19, 'chunk_id': 'pdf_p19_0'}
17 This can mean that AI systems switch from a statistical to rule -based procedure, or that they ask for a human operator before continuing their action .39 It must be ensured that the system will do what it is supposed to do without harming living 

Source 2:
{'source_type': 'eu_ai_act', 'source': 'resources/trustworthy_ai.pdf', 'filename': 'trustworthy_ai.pdf', 'page': 4, 'chunk_id': 'pdf_p4_2'}
AI systems in a way that adhere s to the ethical principles of : respect for h uman autonomy, prevention of h arm, fairness and explicability. Acknowledge and address the potential tensions between these principles.  Pay particular attention to situ

Source 3:
{'source_type': '

#### Step 7: Evaluate Performance

For the query “What does the EU AI Act say about high-risk AI systems?”, the baseline embedding retrieval returned chunks that were broadly related to AI risk, ethics, and legal context, but not direct statements about the EU AI Act itself. Step 3 improved ranking by boosting the chunk that most clearly discussed risk assessment and human oversight, while Step 4 with the cross-encoder was stricter and pushed weaker borderline chunks lower. However, all three methods still retrieved mostly adjacent or thematic content rather than explicit EU AI Act passages on high-risk AI systems. The final RAG answer appropriately stated that the provided context was insufficient to answer the question clearly.
